In [74]:
!pip install -q langchain langchain-google-genai

In [75]:
import os
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
os.environ["GOOGLE_API_KEY"] = "YOUR API KEY"

In [77]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [78]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [79]:
@tool
def check_eligibility(
    cgpa: float,
    required_cgpa: float,
    branch: str,
    eligible_branches: str,
    backlogs: int
) -> str:
    """
    Check whether a student is eligible for a placement opportunity.

    Args:
        cgpa: Student's current CGPA.
        required_cgpa: Minimum CGPA required by the company.
        branch: Student's academic branch.
        eligible_branches: Comma-separated branches accepted by the company.
        backlogs: Number of active backlogs.

    Returns:
        A simple eligibility result explaining whether the student is eligible.

      Make sure that other branches are also eligible only if they are closely related to the eligible_branches
    """

    branches = [
        branch_name.strip().lower()
        for branch_name in eligible_branches.split(",")
    ]

    if (
        cgpa >= required_cgpa
        and branch.lower() in branches
        and backlogs == 0
    ):
        return "Eligible for the placement opportunity."

    reasons = []

    if cgpa < required_cgpa:
        reasons.append("CGPA is below the required CGPA.")

    if branch.lower() not in branches:
        reasons.append("Branch is not eligible.")

    if backlogs > 0:
        reasons.append("Student has active backlogs.")

    return "Not eligible. " + " ".join(reasons)

In [80]:
eligibility_tools = [
    check_eligibility
]

In [81]:
eligibility_agent = create_agent(
    model=llm,
    tools=eligibility_tools,
    system_prompt="""
    You are a placement eligibility agent.

    Check whether the student satisfies the placement requirements.

    Return ONLY this format:

    Eligibility: Eligible / Not Eligible
    Reason: <short reason>

    Do not use markdown.
    Do not add explanations.
    Do not add greetings or conclusions.
    """
)

In [82]:
@tool
def analyze_job_description(job_description: str) -> str:
    """
    Extract important placement requirements from a job description.

    Args:
        job_description: The complete job description provided by a company.

    Returns:
        The job description itself for the agent to analyze into
        role, skills, eligibility requirements, and responsibilities.

         Do not provide explanations, suggestions, or extra information.
    """

    return job_description

In [83]:
job_analysis_tools = [
    analyze_job_description
]

In [84]:
job_analysis_agent = create_agent(
    model=llm,
    tools=job_analysis_tools,
    system_prompt="""
    You are a placement job description analysis agent.

    Analyze the given job description.

    Return ONLY this format:

    Job Role: <role>

    Required Skills:
    <skill 1>
    <skill 2>
    <skill 3>

    Eligibility:
    <eligibility requirements>

    Responsibilities:
    <responsibility 1>
    <responsibility 2>
    <responsibility 3>

    If a section is not mentioned, write:
    Not specified

    Do not use markdown.
    Do not use **.
    Do not add explanations.
    Do not add greetings or conclusions.
    """
)

In [85]:
@tool
def match_skills(
    student_skills: str,
    required_skills: str
) -> str:
    """
    Compare a student's skills with the skills required for a placement role.

    Args:
        student_skills: Comma-separated skills known by the student.
        required_skills: Comma-separated skills required by the company.

    Returns:
        A list of matched skills and missing skills.
    """

    student = {
        skill.strip().lower()
        for skill in student_skills.split(",")
    }

    required = {
        skill.strip().lower()
        for skill in required_skills.split(",")
    }

    matched = student.intersection(required)
    missing = required - student

    return (
        f"Matched skills: {', '.join(sorted(matched)) or 'None'}\n"
        f"Missing skills: {', '.join(sorted(missing)) or 'None'}"
    )

In [86]:
skill_matching_tools = [
    match_skills
]

In [87]:
skill_matching_agent = create_agent(
    model=llm,
    tools=skill_matching_tools,
    system_prompt="""
    You are a placement skill matching agent.

    Compare the student's skills with the required skills.

    Return ONLY this format:

    Matched Skills:
    <skills>

    Missing Skills:
    <skills>

    Do not use markdown.
    Do not add explanations.
    """
)

In [88]:
@tool
def generate_interview_questions(
    role: str,
    skills: str
) -> str:
    """
    Prepare the information needed to generate placement interview questions.

    Args:
        role: The job role the student is applying for.
        skills: Comma-separated technical skills relevant to the role.

    Returns:
        The role and skills that should be used for interview preparation.
    """

    return f"Role: {role}\nSkills: {skills}"

In [89]:
interview_tools = [
    generate_interview_questions
]

In [90]:
interview_agent = create_agent(
    model=llm,
    tools=interview_tools,
    system_prompt="""
    You are a placement interview preparation agent.

    Generate interview questions based on the given role and skills.

    Return ONLY:

    Technical Questions:
    1. <question>
    2. <question>
    3. <question>

    Coding Questions:
    1. <question>
    2. <question>

    Behavioral Questions:
    1. <question>
    2. <question>

    Do not provide answers.
    Do not use markdown.
    Do not add explanations.
    """
)

In [91]:
@tool
def create_prep_plan(
    role: str,
    days: int,
    skills: str
) -> str:
    """
    Prepare the information needed to create a placement preparation plan.

    Args:
        role: The placement role the student is preparing for.
        days: Number of days available for preparation.
        skills: Comma-separated skills required for the role.

    Returns:
        The role, preparation duration, and required skills.
    """

    return (
        f"Role: {role}\n"
        f"Preparation days: {days}\n"
        f"Required skills: {skills}"
    )

In [92]:
prep_tools = [
    create_prep_plan
]

In [93]:
prep_agent = create_agent(
    model=llm,
    tools=prep_tools,
    system_prompt="""
    You are a placement preparation planning agent.

    Create a realistic preparation plan based on the
    role, required skills, and available preparation days.

    Return ONLY:

    Day 1-X:
    <topic>

    Day X-Y:
    <topic>

    Day Y-Z:
    <topic>

    Do not use markdown.
    Do not add explanations.
    """
)

In [94]:
def get_agent_text(response):
    content = response["messages"][-1].content

    if isinstance(content, list):
        return "\n".join(
            block["text"]
            for block in content
            if block.get("type") == "text"
        )

    return content

In [101]:
# --------------------------------------------------
# PREDEFINED PLACEMENT INFORMATION
# --------------------------------------------------

company = "ABC Technologies"

required_cgpa = 8.0
eligible_branches = "CSE, IT, ECE"

role = "SDE-"

required_skills = "Python, SQL, Machine Learning, Statistics"


# --------------------------------------------------
# USER CHOICE
# --------------------------------------------------

choice = input("""
Choose what you want to do:

1. Check placement eligibility
2. Analyze job description
3. Match skills
4. Generate interview questions
5. Create preparation plan

Enter your choice:
""")


# --------------------------------------------------
# 1. CHECK ELIGIBILITY
# --------------------------------------------------

if choice == "1":

    cgpa = input("Enter your CGPA: ")
    branch = input("Enter your branch: ")
    backlogs = input("Enter number of backlogs: ")

    response = eligibility_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
                Check my eligibility for {company}.

                Company requirements:
                Required CGPA: {required_cgpa}
                Eligible branches: {eligible_branches}

                My details:
                CGPA: {cgpa}
                Branch: {branch}
                Backlogs: {backlogs}
                """
            }
        ]
    })

    print("\nResult:")
    print(get_agent_text(response))


# --------------------------------------------------
# 2. ANALYZE JOB DESCRIPTION
# --------------------------------------------------

elif choice == "2":

    job_description = input("""
Paste the job description:

""")

    response = job_analysis_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": job_description
            }
        ]
    })

    print("\nJob Analysis:")
    print(get_agent_text(response))


# --------------------------------------------------
# 3. MATCH SKILLS
# --------------------------------------------------

elif choice == "3":

    student_skills = input(
        "\nEnter your skills (comma-separated): "
    )

    response = skill_matching_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
                Compare my skills with the skills required
                for the {role} role at {company}.

                Required skills:
                {required_skills}

                My skills:
                {student_skills}
                """
            }
        ]
    })

    print("\nSkill Matching Result:")
    print(get_agent_text(response))


# --------------------------------------------------
# 4. GENERATE INTERVIEW QUESTIONS
# --------------------------------------------------

elif choice == "4":

    response = interview_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
                Generate interview questions for the following
                placement role.

                Company: {company}
                Role: {role}
                Required skills: {required_skills}

                Generate questions suitable for a student
                preparing for this placement.
                """
            }
        ]
    })

    print("\nInterview Questions:")
    print(get_agent_text(response))


# --------------------------------------------------
# 5. CREATE PREPARATION PLAN
# --------------------------------------------------

elif choice == "5":

    days = input(
        "\nHow many days do you have for preparation? "
    )

    response = prep_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
                Create a preparation plan for this placement.

                Company: {company}
                Role: {role}
                Required skills: {required_skills}

                I have {days} days available for preparation.

                Create a realistic preparation plan.
                """
            }
        ]
    })

    print("\nPreparation Plan:")
    print(get_agent_text(response))


# --------------------------------------------------
# INVALID CHOICE
# --------------------------------------------------

else:

    print("\nInvalid choice. Please enter a number from 1 to 5.")


Choose what you want to do:

1. Check placement eligibility
2. Analyze job description
3. Match skills
4. Generate interview questions
5. Create preparation plan

Enter your choice:
5

How many days do you have for preparation? 5

Preparation Plan:
Day 1-2:
Python and SQL

Day 3-4:
Machine Learning and Statistics

Day 5:
Mock tests and revision
